# Results overview


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reported_runs/analysis").is_dir())
sys.path.insert(0, str(repo)) if str(repo) not in sys.path else None
from reported_runs.analysis.presentation import load_results, NotebookReport, temporal_table, effect_table, rank_summary, unique_trajectories, label_conditions
from reported_runs.analysis.plotting import trajectory_grid, effect_grid, rank_grid, association_grid
from reported_runs.analysis.comparisons import summarise_iterations
%matplotlib inline

results = load_results()
report = NotebookReport(results, 'analysis')
scales = report.scales
trajectory = label_conditions(results["trajectory"])
sources = results["sources"]
summaries = results["summaries"]
effect_summaries = results["effect_summaries"]
pd.set_option("display.max_columns", 18)
pd.set_option("display.max_rows", 60)

## Snapshot and run status


In [2]:
status = sources.groupby(['dataset', 'study', 'status'], dropna=False).agg(logical_cells=('source_id', 'size'), physical_trajectories=('source_id', 'nunique')).reset_index()
report.table(status, 'analysis_source_status', "")
display(Markdown(f"Snapshot: `{results.metadata['snapshot_id']}`; {results.metadata['completed_physical_runs']} completed physical trajectories; {results.metadata['pending_logical_cells']} pending logical cells."))

,dataset,study,status,logical_cells,physical_trajectories
0,goodreads,goodreads_gender_a300_e804,complete,42,42
1,goodreads,goodreads_gender_mitigation_a300_e804,complete,81,63
2,movielens,movielens_feedback_a300_e804,complete,54,54
3,movielens,movielens_mitigation_a300_e804,complete,45,45
4,movielens,movielens_sampling_outcome_a150_r150_e804,complete,21,21


LaTeX label: `chap:results:tab:analysis_source_status` · [table](../../../reported_runs/analysis/snapshots/results_20261116_goodreads_agent_l0/exports/tables/analysis_source_status.tex) · [numeric CSV](../../../reported_runs/analysis/snapshots/results_20261116_goodreads_agent_l0/exports/tables/analysis_source_status.csv)

Snapshot: `results_20261116_goodreads_agent_l0`; 183 completed physical trajectories; 0 pending logical cells.

In [3]:
pending = sources[sources['status'].ne('complete')]
report.table(pending[['study', 'seed', 'model', 'intervention', 'status']], 'analysis_pending_sources', "")

,study,seed,model,mitigation,status


LaTeX label: `chap:results:tab:analysis_pending_sources` · [table](../../../reported_runs/analysis/snapshots/results_20261116_goodreads_agent_l0/exports/tables/analysis_pending_sources.tex) · [numeric CSV](../../../reported_runs/analysis/snapshots/results_20261116_goodreads_agent_l0/exports/tables/analysis_pending_sources.csv)

## Metric and plotting conventions


## Reproducibility checks


In [4]:
health = results['health'].drop_duplicates('physical_run_id')
report.table(health[['dataset', 'seed', 'physical_run_id', 'events_sha256_verified', 'diagnostics_sha256_verified', 'canonical_opportunities', 'iterations']], 'analysis_verified_sources', "")

,dataset,seed,physical_run_id,events_sha256_verified,diagnostics_sha256_verified,canonical_opportunities,iterations
0,goodreads,42,goodreads_a300_e804_v1__L0_control__agentic_re...,True,True,9000,30
1,goodreads,42,goodreads_a300_e804_v1__L0_control__direct_llm...,True,True,9000,30
2,goodreads,42,goodreads_a300_e804_v1__L0_control__ncm_inspir...,True,True,9000,30
3,goodreads,42,goodreads_a300_e804_v1__L0_control__ncm_inspir...,True,True,9000,30
4,goodreads,42,goodreads_a300_e804_v1__L0_control__position_b...,True,True,9000,30
...,...,...,...,...,...,...,...
178,movielens,44,l4_l5_p18844_seed44_a150_r150_e804_v1__L4_samp...,True,True,2951,30
179,movielens,44,l4_l5_p18844_seed44_a150_r150_e804_v1__L4_samp...,True,True,4500,30
180,movielens,44,l4_l5_p18844_seed44_a150_r150_e804_v1__O0_orga...,True,True,4500,30
181,movielens,44,l4_l5_p18844_seed44_a150_r150_e804_v1__O1_stat...,True,True,4500,30


LaTeX label: `chap:results:tab:analysis_verified_sources` · [table](../../../reported_runs/analysis/snapshots/results_20261116_goodreads_agent_l0/exports/tables/analysis_verified_sources.tex) · [numeric CSV](../../../reported_runs/analysis/snapshots/results_20261116_goodreads_agent_l0/exports/tables/analysis_verified_sources.csv)

## LLM replica agreement across all recorded cases


In [5]:
from reported_runs.analysis.replica_agreement import build_replica_agreement
from reported_runs.analysis.replica_agreement_views import compact_replica_table, TABLE_NAME, TABLE_COLUMN_FORMAT, COUNT_EXPORT_NAME

replica_runs = build_replica_agreement(results['probes'], sources)
replica_table = compact_replica_table(replica_runs)
replica_csv = report.exports / 'data' / COUNT_EXPORT_NAME
replica_csv.parent.mkdir(parents=True, exist_ok=True)
replica_runs.to_csv(replica_csv, index=False)
report.table(replica_table, TABLE_NAME, "", source_frame=replica_runs[replica_runs.replica_status.ne('not_recorded')], column_format=TABLE_COLUMN_FORMAT)
display(Markdown(f'[run CSV]({replica_csv.as_posix()})'))

,Dataset,Model,Condition,Mitigation,Seed 42,Seed 43,Seed 44
0,MovieLens,Direct LLM,L0,None,53.9% (640),53.5% (740),55.6% (698)
1,MovieLens,Direct LLM,L1,None,"58.0% (1,380)","61.3% (1,359)","59.3% (1,368)"
2,MovieLens,Direct LLM,L2,None,"61.0% (3,567)","65.4% (3,797)","62.7% (3,408)"
3,MovieLens,Direct LLM,L2,In-processing,"58.2% (3,692)","63.1% (3,305)","62.1% (3,326)"
4,MovieLens,Direct LLM,L2,Post-processing,"60.5% (3,173)","62.3% (3,561)","60.5% (3,460)"
5,MovieLens,Agent,L0 -- frozen agent,None,"69.3% (4,723)","69.2% (4,663)","68.8% (4,685)"
6,MovieLens,Agent,L1 -- frozen agent,None,"69.0% (7,864)","70.8% (7,835)","70.4% (7,692)"
7,MovieLens,Agent,L2 -- frozen agent,None,"59.0% (8,059)","59.4% (8,037)","61.6% (8,152)"
8,MovieLens,Agent,L3 -- no reflection,None,"58.4% (8,053)","59.8% (8,033)","62.0% (8,018)"
9,MovieLens,Agent,L3 -- fixed profile,None,"61.2% (7,935)","61.0% (8,009)","63.9% (8,006)"


LaTeX label: `chap:results:tab:analysis_llm_replica_agreement_all_cases` · [table](../../../reported_runs/analysis/snapshots/results_20261116_goodreads_agent_l0/exports/tables/analysis_llm_replica_agreement_all_cases.tex) · [numeric CSV](../../../reported_runs/analysis/snapshots/results_20261116_goodreads_agent_l0/exports/tables/analysis_llm_replica_agreement_all_cases.csv)

[run CSV](../../../reported_runs/analysis/snapshots/results_20261116_goodreads_agent_l0/exports/data/analysis_llm_replica_agreement_runs.csv)

## Suggested main-text figures


## Exports and refreshing


In [6]:
display(Markdown(f"**Exports:** `{report.exports}`\n\n**Metric version:** `{results.metadata['metric_version']}`\n\n**Draft source:** `{results.metadata['draft_path']}`"))

**Exports:** `../../../reported_runs\analysis\snapshots\results_20261116_goodreads_agent_l0\exports`

**Metric version:** `final-rq-metrics-v1-dcg-one-based`

**Draft source:** `[thesis source not included]`

## Portable replay scope
